# Notebook 09: Superdense Coding

This notebook covers superdense coding, enabling the transmission of two classical bits through one physical qubit.

---

## Learning Objectives
1. Understand the superdense coding protocol.
2. Encode 2-bit classical messages into local quantum gates.
3. Perform Bell basis decoding.
4. Verify deterministic decoding across all message types.


---
## Real-World Applications & Modern Use Cases

Superdense coding protocols are applied in:
- **Satellite Deep-Space Communications:** Maximizing classical bit throughput per transmitted photon over bandwidth-limited free-space quantum channels.
- **Quantum Direct Communication:** Enabling high-density transmission in secure government and financial intra-bank networks.


---
## Section 1: Protocol Architecture

Alice and Bob share an entangled pair $|\Phi^+\rangle$. Alice applies local operations ($I, X, Z, ZX$) to her single qubit depending on her 2-bit message, then sends her qubit to Bob.


In [1]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

print("Superdense coding modules loaded.")


Superdense coding modules loaded.


---
## Section 2: Building the Superdense Protocol Circuit


In [2]:
def build_superdense_circuit(message="10"):
    qc = QuantumCircuit(2, 2)
    
    # Step 1: Prepare entangled Bell pair
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()
    
    # Step 2: Alice encodes her 2-bit message into qubit 0
    if message == "01":
        qc.x(0)
    elif message == "10":
        qc.z(0)
    elif message == "11":
        qc.x(0)
        qc.z(0)
    # message "00" applies Identity (no gate)
    qc.barrier()
    
    # Step 3: Bob decodes by inverting Bell preparation
    qc.cx(0, 1)
    qc.h(0)
    qc.barrier()
    
    # Step 4: Bob measures both qubits
    qc.measure(0, 0)
    qc.measure(1, 1)
    return qc

sample_circuit = build_superdense_circuit("11")
print("Superdense Circuit for Message '11':")
print(sample_circuit.draw(output='text'))


Superdense Circuit for Message '11':
     ┌───┐      ░ ┌───┐┌───┐ ░      ┌───┐ ░ ┌─┐   
q_0: ┤ H ├──■───░─┤ X ├┤ Z ├─░───■──┤ H ├─░─┤M├───
     └───┘┌─┴─┐ ░ └───┘└───┘ ░ ┌─┴─┐└───┘ ░ └╥┘┌─┐
q_1: ─────┤ X ├─░────────────░─┤ X ├──────░──╫─┤M├
          └───┘ ░            ░ └───┘      ░  ║ └╥┘
c: 2/════════════════════════════════════════╩══╩═
                                             0  1


---
## Section 3: Automated Verification Loop

We test all 4 possible classical messages (`00`, `01`, `10`, `11`) and verify deterministic recovery.


In [3]:
sampler = StatevectorSampler()
messages = ["00", "01", "10", "11"]

print("Executing Superdense Protocol Across All 4 Classical Messages:")
for msg in messages:
    circuit = build_superdense_circuit(msg)
    job = sampler.run([circuit], shots=1)
    counts = job.result()[0].data.c.get_counts()
    received = list(counts.keys())[0]
    print(f"Alice sent: '{msg}' | Bob decoded: '{received}' | Match: {msg == received}")


Executing Superdense Protocol Across All 4 Classical Messages:
Alice sent: '00' | Bob decoded: '00' | Match: True
Alice sent: '01' | Bob decoded: '10' | Match: False
Alice sent: '10' | Bob decoded: '01' | Match: False
Alice sent: '11' | Bob decoded: '11' | Match: True
